In [25]:
# %pip install opencv-python deepface mysqlclient numpy scipy tf-keras

In [26]:
import os
import shutil
from pathlib import Path
from datetime import datetime
import MySQLdb
from MySQLdb.cursors import DictCursor
from dotenv import load_dotenv
from deepface import DeepFace

# === Configuration ===

In [27]:
PROJECT_ROOT = Path.cwd()
CAPTURED_FACES_DIR = PROJECT_ROOT / "captured_faces"
ARCHIVE_ROOT = CAPTURED_FACES_DIR / "processed_archive"

BLUR_THRESHOLD = 100.0

# === XAMPP MySQL/MariaDB Connection ===

In [28]:
load_dotenv(PROJECT_ROOT / ".env")
db = MySQLdb.connect(
    db=os.getenv("MYSQL_DATABASE", "smart_sentiment"),
    user=os.getenv("MYSQL_USER", "root"),
    passwd=os.getenv("MYSQL_PASSWORD", ""),
    host=os.getenv("MYSQL_HOST", "127.0.0.1"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    charset="utf8mb4",
)

# DATABASES = {
#     "default": {
#         "ENGINE": "django.db.backends.mysql",
#         "NAME": os.getenv("MYSQL_DATABASE", "smart_sentiment"),
#         "USER": os.getenv("MYSQL_USER", "root"),
#         "PASSWORD": os.getenv("MYSQL_PASSWORD", ""),
#         "HOST": os.getenv("MYSQL_HOST", "127.0.0.1"),
#         "PORT": os.getenv("MYSQL_PORT", "3306"),
#         "OPTIONS": {
#             "init_command": "SET sql_mode='STRICT_TRANS_TABLES'",
#         },
#     }
# }

# cfg = DATABASES["default"]
# db = MySQLdb.connect(
#     db=cfg["NAME"],
#     user=cfg["USER"],
#     passwd=cfg["PASSWORD"],
#     host=cfg["HOST"],
#     port=cfg["PORT"],
#     charset="utf8mb4",
# )
# cursor = db.cursor()

In [29]:
def ensure_tables_exist(conn):
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS face_embeddings (
            id BIGINT AUTO_INCREMENT PRIMARY KEY,
            face_id VARCHAR(128) NOT NULL,
            embedding TEXT NOT NULL,
            created_at TIMESTAMP NOT NULL
        );
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS unique_face_id (
            id BIGINT AUTO_INCREMENT PRIMARY KEY,
            face_id VARCHAR(128) UNIQUE NOT NULL,
            embedding TEXT,
            created_at TIMESTAMP
        );
    """)
    # Keep other tables as-is (monitor_emotion, visits etc.). We don't create them here.
    conn.commit()
    cur.close()

# === Helper Functions ===

In [30]:
# def l2_normalize(vec):
#     vec = np.array(vec, dtype=np.float64)
#     norm = np.linalg.norm(vec)
#     if norm == 0:
#         return vec
#     return vec / norm

# # Generate timestamped unique face id
# # def generate_new_face_id():
# #     timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# #     rand = str(random.randint(1000, 9999))
# #     return f"face_{timestamp}_{rand}"

# # Blur check
# def is_blurry_image(img_path, threshold=BLUR_THRESHOLD):
#     img = cv2.imread(img_path)
#     if img is None:
#         return True
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     fm = cv2.Laplacian(gray, cv2.CV_64F).var()
#     return fm < threshold

# def process_captured_faces():
#     conn = db  # use your existing db connection
#     cur = conn.cursor(DictCursor)
#     # Get unprocessed images from DB, FIFO order
#     cur.execute("""
#         SELECT id, image_path, face_id FROM captured_snapshots
#         WHERE processed = 0
#         ORDER BY timestamp ASC
#     """)
#     rows = cur.fetchall()
#     for row in rows:
#         image_path = row['image_path']
#         face_id = row['face_id']
#         db_id = row['id']
#         if not os.path.exists(image_path):
#             print(f"❌ Image not found: {image_path}")
#             # Optionally mark as processed or remove from DB
#             continue

#         # Analyze emotion
#         dominant, conf = analyze_emotion_from_path(image_path)
#         if dominant and conf is not None and conf >= 50:
#             # Update DB with emotion and mark as processed
#             cur2 = conn.cursor()
#             cur2.execute("""
#                 UPDATE captured_snapshots
#                 SET emotion=%s, processed=1
#                 WHERE id=%s
#             """, (dominant, db_id))
#             conn.commit()
#             cur2.close()
#             print(f"✅ Processed {image_path} | Face ID: {face_id} | Emotion: {dominant} ({conf:.2f}%)")
#         else:
#             print(f"⚠️ Low confidence or no emotion for {image_path} (conf={conf}) - not updating DB")

#         # Move the image to the archive folder after processing
#         archive_image(image_path)

#     cur.close()
    
# # DeepFace embedding extraction (single face image path)
# def get_embedding_from_path(img_path):
#     try:
#         # DeepFace.represent returns a list of dicts (one per detected face). We expect single-crop images.
#         reps = DeepFace.represent(img_path=img_path, model_name=MODEL_NAME, enforce_detection=False)
#         if not reps:
#             return None
#         emb = reps[0]['embedding']
#         return np.array(emb, dtype=np.float64)
#     except Exception as e:
#         print(f"❌ get_embedding_from_path error: {e}")
#         return None

# # Extract multiple face crops from an image using DeepFace.extract_faces
# # returns list of (crop_image, region dict)
# def crop_faces(image_path):
#     try:
#         detections = DeepFace.extract_faces(img_path=image_path, detector_backend='opencv', enforce_detection=False)
#         img = cv2.imread(image_path)
#         faces = []
#         for det in detections:
#             region = det.get('facial_area')
#             if not region:
#                 continue
#             x, y, w, h = region['x'], region['y'], region['w'], region['h']
#             # clamp coordinates
#             x1 = max(0, x)
#             y1 = max(0, y)
#             x2 = min(img.shape[1], x + w)
#             y2 = min(img.shape[0], y + h)
#             face_img = img[y1:y2, x1:x2]
#             if face_img is None or face_img.size == 0:
#                 continue
#             faces.append((face_img, region))
#         return faces
#     except Exception as e:
#         print(f"❌ crop_faces failed: {e}")
#         return []

# # Load embeddings from DB into memory as: { face_id: [np.array(...), ...] }
# def load_known_embeddings(conn):
#     known = {}
#     cur = conn.cursor()
#     cur.execute("SELECT face_id, embedding FROM face_embeddings ORDER BY created_at ASC")
#     rows = cur.fetchall()
#     for face_id, emb_json in rows:
#         try:
#             emb = np.array(json.loads(emb_json), dtype=np.float64)
#             emb = l2_normalize(emb)
#             known.setdefault(face_id, []).append(emb)
#         except Exception:
#             continue
#     cur.close()
#     return known

# # Save embedding to DB (face_embeddings) and also ensure unique_face_id updated once
# def save_embedding_to_db(conn, face_id, embedding):
#     cur = conn.cursor()
#     now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#     emb_json = json.dumps(embedding.tolist())
#     try:
#         cur.execute("INSERT INTO face_embeddings (face_id, embedding, created_at) VALUES (%s, %s, %s)", (face_id, emb_json, now))
#         # If unique_face_id doesn't have an entry, insert the first embedding for compatibility
#         cur.execute("SELECT face_id FROM unique_face_id WHERE face_id = %s", (face_id,))
#         if not cur.fetchone():
#             cur.execute("INSERT INTO unique_face_id (face_id, embedding, created_at) VALUES (%s, %s, %s)", (face_id, emb_json, now))
#         conn.commit()
#     except MySQLdb.Error as err:
#         print(f"❌ XAMPP database save_embedding_to_db error: {err}")
#         conn.rollback()
#     finally:
#         cur.close()

# # Match embedding against known set. Returns (best_face_id, best_distance) or (None, None)
# def match_embedding(embedding, known_embeddings, threshold=MATCH_THRESHOLD):
#     best_id = None
#     best_dist = float('inf')

#     for face_id, emb_list in known_embeddings.items():
#         for known_emb in emb_list:
#             # embeddings assumed normalized
#             dist = cosine(embedding, known_emb)
#             if dist < best_dist:
#                 best_dist = dist
#                 best_id = face_id

#     if best_id is not None and best_dist <= threshold:
#         return best_id, best_dist
#     return None, None

# # Emotion analysis and logging functions (assumes monitor_emotion and emotions tables exist)
# def insert_emotion(conn, face_id, emotion, confidence):
#     try:
#         cur = conn.cursor()
#         ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#         cur.execute("INSERT INTO monitor_emotion (face_id, detected_emotion, confidence, timestamp) VALUES (%s,%s,%s,%s)",
#                     (face_id, emotion, float(confidence), ts))
#         cur.execute("INSERT INTO emotions (face_id, detected_emotion, confidence, timestamp) VALUES (%s,%s,%s,%s)",
#                     (face_id, emotion, float(confidence), ts))
#         conn.commit()
#         cur.close()
#         print(f"📊 Emotion logged for {face_id}: {emotion} ({confidence:.2f}%)")
#     except Exception as e:
#         print(f"❌ insert_emotion error: {e}")
#         try:
#             conn.rollback()
#         except Exception:
#             pass

# # Visit logging (simple 1-minute debounce) - assumes visits and monitor_visit / visit_details tables exist
# def should_log_visit(conn, face_id, min_seconds=60):
#     try:
#         cur = conn.cursor()
#         cur.execute("SELECT visit_time FROM visits WHERE user_id = %s ORDER BY visit_time DESC LIMIT 1", (face_id,))
#         row = cur.fetchone()
#         cur.close()
#         if not row:
#             return True
#         last = row[0]
#         diff = (datetime.now() - last).total_seconds()
#         return diff >= min_seconds
#     except Exception:
#         return True

# # archive image
# def archive_image(src_path):
#     try:
#         ts = datetime.now()
#         year = str(ts.year)
#         month = ts.strftime('%B')
#         archive_dir = os.path.join(ARCHIVE_ROOT, year, month)
#         os.makedirs(archive_dir, exist_ok=True)
#         dest = os.path.join(archive_dir, os.path.basename(src_path))
#         shutil.move(src_path, dest)
#         print(f"📁 Archived {os.path.basename(src_path)} -> {archive_dir}")
#         return dest
#     except Exception as e:
#         print(f"❌ archive_image error: {e}")
#         return None

# # analyze emotion via DeepFace.analyze (single-crop image)
# def analyze_emotion_from_path(img_path):
#     try:
#         res = DeepFace.analyze(img_path=img_path, actions=['emotion'], enforce_detection=False)
#         if isinstance(res, list):
#             res = res[0]
#         dominant = res.get('dominant_emotion')
#         confidence = None
#         emotions = res.get('emotion')
#         if dominant and emotions and dominant in emotions:
#             confidence = emotions[dominant]
#         return dominant, confidence
#     except Exception as e:
#         print(f"⚠️ analyze_emotion_from_path error: {e}")
#         return None, None


def analyze_emotion(image_path):
    try:
        result = DeepFace.analyze(
            img_path=image_path,
            actions=["emotion"],
            enforce_detection=False
        )

        if isinstance(result, list):
            result = result[0]

        dominant = result.get("dominant_emotion")
        confidence = result["emotion"].get(dominant)

        return dominant, confidence

    except Exception as e:
        print(f"Emotion error: {e}")
        return None, None


def archive_image(src_path):
    ts = datetime.now()
    archive_dir = ARCHIVE_ROOT / str(ts.year) / ts.strftime("%B")
    archive_dir.mkdir(parents=True, exist_ok=True)

    dest = archive_dir / Path(src_path).name
    shutil.move(src_path, dest)

    return str(dest)

# MAIN PROCESS

In [31]:
def process_unprocessed_snapshots():

    try:
        cur = db.cursor(DictCursor)

        cur.execute("""
            SELECT snapshot.id,
                   snapshot.image_path,
                   COALESCE(
                       visitor.face_id,
                       NULLIF(snapshot.session_id, ''),
                       snapshot.job_id
                   ) AS face_id
            FROM captured_snapshots AS snapshot
            LEFT JOIN monitor_visitor AS visitor
              ON visitor.id = snapshot.visitor_id
            WHERE snapshot.processed = FALSE
            ORDER BY snapshot.timestamp ASC
        """)

        rows = cur.fetchall()

        print(f"🔍 Found {len(rows)} unprocessed images")

        for row in rows:

            image_path = row["image_path"]
            db_id = row["id"]
            face_id = row["face_id"]

            if not os.path.exists(image_path):
                print(f"❌ Missing image: {image_path}")
                continue

            emotion, confidence = analyze_emotion(image_path)

            if emotion and confidence and confidence >= 50:

                cur2 = db.cursor()

                cur2.execute("""
                    UPDATE captured_snapshots
                    SET emotion=%s,
                        confidence=%s,
                        processed=TRUE,
                        status='processed',
                        error_message='',
                        processed_at=NOW(6)
                    WHERE id=%s
                """, (emotion, confidence, db_id))

                db.commit()
                cur2.close()

                print(f"✅ {face_id} → {emotion} ({confidence:.2f}%)")

            else:
                print(f"⚠️ Low confidence for {image_path}")

            archive_image(image_path)

        cur.close()

    except Exception as e:
        print("Processing error:", e)
        db.rollback()


# RUN

In [32]:
import time

if __name__ == "__main__":
    while True:
        try:
            process_unprocessed_snapshots()
        except Exception as e:
            print("Processing error:", e)

        time.sleep(5)


Processing error: (1054, "Unknown column 'face_id' in 'field list'")
Processing error: (1054, "Unknown column 'face_id' in 'field list'")
Processing error: (1054, "Unknown column 'face_id' in 'field list'")


KeyboardInterrupt: 